In [ ]:
!pip install unsloth trl transformers accelerate peft bitsandbytes wandb matplotlib datasets -q
!pip install git+https://github.com/meta-pytorch/OpenEnv.git -q
!git clone https://github.com/ShivenduShivu/ProcureRL.git
%cd ProcureRL
!pip install -e . -q


In [ ]:
import json
import os
import torch
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
from agenticpay.openenv_adapter.procure_env_extended import ProcureEnvExtended
from training.prompt_builder import format_for_trl
from training.evaluate import evaluate_model, print_evaluation_report
from training.plot_results import save_before_after_plot, save_training_plots


In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
)


In [ ]:
print('Running BASELINE evaluation...')
baseline_metrics = evaluate_model(model, tokenizer, n_episodes=30, difficulty='easy')
print_evaluation_report(baseline_metrics, label='BASELINE (Before Training)')
os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2)


In [ ]:
env_for_reward = ProcureEnvExtended(difficulty='easy')

def compute_reward_for_grpo(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        try:
            obs, _ = env_for_reward.reset(seed=hash(prompt) % 10000)
            _, reward, _, _, _ = env_for_reward.step(completion)
            rewards.append(reward)
        except Exception:
            rewards.append(0.0)
    return rewards


In [ ]:
env_for_data = ProcureEnvExtended(difficulty='easy')
training_prompts = []
for i in range(500):
    obs, _ = env_for_data.reset(seed=i)
    messages = format_for_trl(obs)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    training_prompts.append({'prompt': prompt})
train_dataset = Dataset.from_list(training_prompts)


In [ ]:
training_log = []
grpo_config = GRPOConfig(
    output_dir='./procurerl_checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=8,
    max_new_tokens=256,
    temperature=0.8,
    logging_steps=10,
    save_steps=100,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[compute_reward_for_grpo],
    args=grpo_config,
    train_dataset=train_dataset,
)
trainer.train()


In [ ]:
print('Running POST-TRAINING evaluation...')
trained_metrics = evaluate_model(model, tokenizer, n_episodes=30, difficulty='easy')
print_evaluation_report(trained_metrics, label='TRAINED (After GRPO)')
with open('results/trained_metrics.json', 'w') as f:
    json.dump(trained_metrics, f, indent=2)


In [ ]:
save_before_after_plot(baseline_metrics, trained_metrics)
print('Improvement in deal rate:', trained_metrics['deal_rate'] - baseline_metrics['deal_rate'])
print('Improvement in mean reward:', trained_metrics['mean_episode_reward'] - baseline_metrics['mean_episode_reward'])


In [ ]:
model.save_pretrained_merged(
    'procurerl_model_merged',
    tokenizer,
    save_method='merged_16bit',
)
print('Model saved.')
